# Chapter 6 - Working with different types of data

In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/27 07:58:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


In [3]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/data/retail-data/by-day/2010-12-01.csv")
)
df.printSchema()
df.createOrReplaceTempView("dfTable")

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [5]:
# How to convert to Spark types
import pyspark.sql.functions as F
df.select(F.lit(5), F.lit("five"), F.lit(5.0))

DataFrame[5: int, five: string, 5.0: double]

In [8]:
# Filtering
df.where(F.col("InvoiceNo") != 536365).select("InvoiceNo", "Description").show(5, False)

# Cleanest way
df.where("InvoiceNo <> 536365").show(5, False)
df.where("InvoiceNo = 536365").show(5, False)

+---------+-----------------------------+
|InvoiceNo|Description                  |
+---------+-----------------------------+
|536366   |HAND WARMER UNION JACK       |
|536366   |HAND WARMER RED POLKA DOT    |
|536367   |ASSORTED COLOUR BIRD ORNAMENT|
|536367   |POPPY'S PLAYHOUSE BEDROOM    |
|536367   |POPPY'S PLAYHOUSE KITCHEN    |
+---------+-----------------------------+
only showing top 5 rows
+---------+---------+-----------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                  |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------+--------+-------------------+---------+----------+--------------+
|536366   |22633    |HAND WARMER UNION JACK       |6       |2010-12-01 08:28:00|1.85     |17850.0   |United Kingdom|
|536366   |22632    |HAND WARMER RED POLKA DOT    |6       |2010-12-01 08:28:00|1.85     |17850.0   |United Kingdom|
|536367   |84

In [14]:
from pyspark.sql.functions import instr
priceFilter = F.col("UnitPrice") > 600
descripFilter = instr(df.Description,"POSTAGE") >= 1
df.where(df.StockCode.isin("DOT")).where(priceFilter | descripFilter).show()


# other example
DOTCodeFilter = df.StockCode == "DOT"
priceFilter = df.UnitPrice > 600
descripFilter = F.instr(df.Description, "POSTAGE") >= 1

(
    df
    .withColumn("isExpensive", DOTCodeFilter & (priceFilter | descripFilter))
    .where("isExpensive") # We can just filter by a column name if the columns is a boolean
    .select("unitPrice", "isExpensive").show(5)
)

# We can also
(
    df.withColumn("isExpensive", F.expr("NOT UnitPrice <= 250"))
    .where("isExpensive")
    .select("Description", "UnitPrice").show(5)
)

# If the columns can contains nulls, we need to use the NullSafe approach
(
    df.where(df.Description.eqNullSafe("hello")).show()
)

+---------+---------+--------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|   Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------+--------+-------------------+---------+----------+--------------+
|   536544|      DOT|DOTCOM POSTAGE|       1|2010-12-01 14:32:00|   569.77|      NULL|United Kingdom|
|   536592|      DOT|DOTCOM POSTAGE|       1|2010-12-01 17:06:00|   607.49|      NULL|United Kingdom|
+---------+---------+--------------+--------+-------------------+---------+----------+--------------+

+---------+-----------+
|unitPrice|isExpensive|
+---------+-----------+
|   569.77|       true|
|   607.49|       true|
+---------+-----------+

+--------------+---------+
|   Description|UnitPrice|
+--------------+---------+
|DOTCOM POSTAGE|   569.77|
|DOTCOM POSTAGE|   607.49|
+--------------+---------+

+---------+---------+-----------+--------+-----------+---------+----------+----

In [7]:
# Working with numbers
import pyspark.sql.functions as F
fabricatedQuantity = F.pow(F.col("Quantity") * F.col("UnitPrice"), 2) + 5
df.select(F.expr("CustomerId"), fabricatedQuantity.alias("realQuantity")).show(2)

# We can also do the same in SQL
df.selectExpr(
    "CustomerId",
    "(POWER((Quantity * UnitPrice), 2.0) +5) as realQuantity"
).show(2)


# How to round numbers. By default, the round functions rounds up, when in the middle of 2 numbers,
# if we want to round down, we can use the bround
df.select(F.round(F.lit(2.5)), F.bround(F.lit(2.5))).show(2)

+----------+------------------+
|CustomerId|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 2 rows
+----------+------------------+
|CustomerId|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 2 rows
+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 2 rows


# What is a correlation?
Correlation is a way for us to measure if two things happen together.
In an easy way:
* If one thing goes up, and the other also goes up, we have a positive correlation;
* If one thing goes up, and the other goes down, we have a negative correlation;
* If we can't find a pattern between them, there is no correlation.

**Important:**
Correlation does not equal causation.
Meaning, two things can be related but one does not cause the other.

The Pearson Correlation it's one metric that is used to quantify the linear relation between 2 variables.
* The value can go from: -1 to +1;
* +1: Perfect positive correlation
* -1: Perfect negative correlation
* 0: No correlation

In [9]:
# How to measure correlation of two columns?

df.stat.corr("Quantity", "UnitPrice")
df.select(F.corr("Quantity", "UnitPrice")).show()


# Another cool feature is to describe the summary statistics for a columns or a set of columns
df.describe().show()

+-------------------------+
|corr(Quantity, UnitPrice)|
+-------------------------+
|     -0.04112314436835551|
+-------------------------+



26/08/25 10:34:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|summary|        InvoiceNo|         StockCode|         Description|          Quantity|         UnitPrice|        CustomerID|       Country|
+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|  count|             3108|              3108|                3098|              3108|              3108|              1968|          3108|
|   mean| 536516.684944841|27834.304044117645|                NULL| 8.627413127413128| 4.151946589446603|15661.388719512195|          NULL|
| stddev|72.89447869788873|17407.897548583845|                NULL|26.371821677029203|15.638659854603892|1854.4496996893627|          NULL|
|    min|           536365|             10002| 4 PURPLE FLOCK D...|               -24|               0.0|           12431.0|     Australia|
|    max|          C

In [14]:
# How to use the stat package to do simple analysis on our dataframe
# What does the Quantile mean? It calculates the number on the 0.5 quantile, means, the avg value
colName = "UnitPrice"
quantileProbs = [0.5]
relError = 0.05
df.stat.approxQuantile("UnitPrice", quantileProbs, relError)

df.stat.crosstab("StockCode", "Quantity")

# We can also add a unique ID for each row by using the monotonically_increasing_id
df.select(F.monotonically_increasing_id()).show(2)

+-----------------------------+
|monotonically_increasing_id()|
+-----------------------------+
|                            0|
|                            1|
+-----------------------------+
only showing top 2 rows


In [ ]:
# Working with Strings
# capitalize every word after a space
df.select(F.initcap(F.col("Description"))).show(2, False)

# We can also add or remove spaces between strings
(df
    .select(F.ltrim(F.lit(" HELLO ")).alias("ltrim"),
            F.rtrim(F.lit(" HELLO ")).alias("rtrim"),
            F.trim(F.lit(" HELLO ")).alias("trim"),
            F.lpad(F.lit("HELLO"), 3, " ").alias("lp"),
            F.rpad(F.lit("HELLO"), 10," ").alias("rp")).show(2))

+----------------------------------+
|initcap(Description)              |
+----------------------------------+
|White Hanging Heart T-light Holder|
|White Metal Lantern               |
+----------------------------------+
only showing top 2 rows
+------+------+-----+---+----------+
| ltrim| rtrim| trim| lp|        rp|
+------+------+-----+---+----------+
|HELLO | HELLO|HELLO|HEL|HELLO     |
|HELLO | HELLO|HELLO|HEL|HELLO     |
+------+------+-----+---+----------+
only showing top 2 rows


In [ ]:
# Regex_replace. Replace any of the colors by the string COLOR
import pyspark.sql.functions as F
regex_string = "BLACK|WHITE|RED|GREEN|BLUE"

(
    df
    .select(
        F.regexp_replace(F.col("Description"), regex_string, "COLOR").alias("color_clean"),
        F.col("Description")
    ).show(2)
)

# Replace characters, it matches one char, like T to the number 1, E to 3, etc.
(
    df
    .select(
        F.translate(F.col("Description"), "LEET", "1337"),
        F.col("Description")
    ).show(2)
)

# How to get the first mentioned color
extract_str = "(BLACK|WHITE|RED|GREEN|BLUE)"
(
    df
    .select(
        F.regexp_extract(F.col("Description"), extract_str, 1).alias("color_clean"),
        F.col("Description")
    ).show(2)
)

# How to check if some string exists? We can use the contains
containsBlack = F.instr(F.col("Description"), "BLACK") >= 1
containsWhite = F.instr(F.col("Description"), "WHITE") >= 1

(
    df
    .withColumn("hasSimpleColor", containsBlack | containsWhite)
    .where("hasSimpleColor")
    .select("Description")
    .show(3)
)



# How to do this to multiple values?
simpleColors = ["black", "white", "red", "green", "blue"] # define the colors that we want to check
def color_locator(column, color_string):
    # Returns a bool column that specifies if some color exists on the column or not
    return F.locate(
        color_string.upper(), 
        column
    ).cast("boolean").alias("is_" + color_string)

selectedColumns = [color_locator(df.Description, color) for color in simpleColors]
selectedColumns.append(F.expr("*")) # get all possible columns

(
    df
    .select(*selectedColumns)
    .where("is_white or is_red")
    .select("Description").show(3)
)

+--------------------+--------------------+
|         color_clean|         Description|
+--------------------+--------------------+
|COLOR HANGING HEA...|WHITE HANGING HEA...|
| COLOR METAL LANTERN| WHITE METAL LANTERN|
+--------------------+--------------------+
only showing top 2 rows
+----------------------------------+--------------------+
|translate(Description, LEET, 1337)|         Description|
+----------------------------------+--------------------+
|              WHI73 HANGING H3A...|WHITE HANGING HEA...|
|               WHI73 M37A1 1AN73RN| WHITE METAL LANTERN|
+----------------------------------+--------------------+
only showing top 2 rows
+-----------+--------------------+
|color_clean|         Description|
+-----------+--------------------+
|      WHITE|WHITE HANGING HEA...|
|      WHITE| WHITE METAL LANTERN|
+-----------+--------------------+
only showing top 2 rows
+--------------------+
|         Description|
+--------------------+
|WHITE HANGING HEA...|
| WHITE METAL 

## Working with dates and Timestamps
Spark supports 2 types:
* dates - Focus exclusively on calendar dates
* timestamps - Which include both date and time information

**TimestampType:** Spark's TimestampType class supports only second-level precision, which means that if we are going to work with milliseconds or microseconds, you'll need to work around this problem by potentially operating on them as longs. Any more precision when coercing to a TimestampType will be removed.

At the end of the day, Spark is working with Java dates and timestamps and therefore conform to those standards.

In [20]:
# Working with Dates and Timestamps

dateDf = (
    spark
    .range(10)
    .withColumn("today", F.current_date())
    .withColumn("now", F.current_timestamp())
)

dateDf.createOrReplaceTempView("dateTable")
dateDf.printSchema()

# Let's remove 5 days
(
    dateDf
    .select(
        F.date_sub(F.col("today"), 5),
        F.date_add(F.col("today"), 5)
    ).show(1)
)


# Let's calculate the difference between 2 dates
(
    dateDf
    .withColumn("week_ago", F.date_sub(F.col("today"), 7))
    .select(
        F.datediff(F.col("week_ago"), F.col("today"))
    ).show(1)
)

(
    dateDf
    .select(
        F.to_date(F.lit("2016-01-01")).alias("start"),
        F.to_date(F.lit("2017-05-22")).alias("end")
    )
    .select(
        F.months_between(F.col("start"), F.col("end"))
    ).show(1)
)

# when we use the to_date function, we should always specify the date format expected
# Super IMPORTANT: Spark will not thrown an error if it cannot parse the date; rather it will just return null.
# This can be a bit tricky in larger pipelines because you might be expecting your data in one format and getting it in another.

# Here is an example
(
    dateDf
    .select(
        #F.to_date(F.lit("2016-20-12")), # It throws an error now, but in some version will not. And it will give you a null
        F.to_date(F.lit("2017-12-11"))
    ).show(1)
)


# How to fix this issue?
dateFormat = "yyyy-dd-MM"
cleanDateDf = (
    spark
    .range(1)
    .select(
        F.to_date(F.lit("2017-12-11"), dateFormat).alias("date"),
        F.to_date(F.lit("2017-20-12"), dateFormat).alias("date2")
    )
)
cleanDateDf.createOrReplaceTempView("dateTable2")

# We can also use the to_timestamp, which always require the format to be specified
(
    cleanDateDf.select(
        F.to_timestamp(F.col("date"), dateFormat)
    ).show()
)

# If we have everything well formatted, is quite easy to compare between dates
(
    cleanDateDf.filter(F.col("date2") > F.lit("2017-12-12")).show()
)

root
 |-- id: long (nullable = false)
 |-- today: date (nullable = false)
 |-- now: timestamp (nullable = false)

+------------------+------------------+
|date_sub(today, 5)|date_add(today, 5)|
+------------------+------------------+
|        2026-08-21|        2026-08-31|
+------------------+------------------+
only showing top 1 row
+-------------------------+
|datediff(week_ago, today)|
+-------------------------+
|                       -7|
+-------------------------+
only showing top 1 row
+--------------------------------+
|months_between(start, end, true)|
+--------------------------------+
|                    -16.67741935|
+--------------------------------+
only showing top 1 row
+-------------------+
|to_date(2017-12-11)|
+-------------------+
|         2017-12-11|
+-------------------+
only showing top 1 row
+------------------------------+
|to_timestamp(date, yyyy-dd-MM)|
+------------------------------+
|           2017-11-12 00:00:00|
+------------------------------+

+--

## Working with nulls in Data
As a best practice, you should always use nulls to represent missing or empty data in your DataFrames. Spark can optimize working with null values more than it can if you use empty Strings or other values.

**WARNING**
nullable=False in a pyspark StructType is schema metadata, not a general runtime constraint. It may be validated when creating a DataFrame from local python data. However, Spark transformations, expressions, joins, and external writes can still produce or contain NULL values despite the metadata. Therefore, we can't rely on the nullable=False for the data-quality enforcement.

Not the same thing as delta tables though. Delta tables null=False does enforce that some column cannot be null on write.

In [5]:
import pyspark.sql.functions as F
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/data/retail-data/by-day/2010-12-01.csv")
)
df.printSchema()
df.createOrReplaceTempView("dfTable")

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [11]:
# We can 2 things with null values, we can drop the nulls or fill them with a value

# Coalesce
# Returns the first non null value. Requires all types to be the same
(
    df
    .select(
        F.coalesce(
            F.col("Description"),
            F.col("CustomerId").cast("string")
        )
    ).show(1)
)

# ifNull, nullIf, nvl and nvl2
# ifNull allows you to select second value if the first is null and defaults to the first.


# Drop
df.na.drop()
# ANY - Specifying any drops a row if some column of a row is null
df.na.drop("any")
# all - Drop the row only if all columns are null
df.na.drop("all")
# We can also apply this to a certain sets of columns
df.na.drop("all", subset=["StockCode", "InvoiceNo"])

# fill - Allow us to fill one or more columns with a set of values
df.na.fill("All null values become this string")
# We can also pass an array of columns
df.na.fill("all", subset=["StockCode", "InvoiceNo"])
# We can also do this with a map
fill_cols_vals = {
    "StockCode": 5,
    "Description": "No Value"
}
df.na.fill(fill_cols_vals)

# Replace - Allow is to replace all values in a certain column according to their current value
df.na.replace([""], ["UNKNOWN"], "Description")


+-------------------------------------------------+
|coalesce(Description, CAST(CustomerId AS STRING))|
+-------------------------------------------------+
|                             WHITE HANGING HEA...|
+-------------------------------------------------+
only showing top 1 row


DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

In [26]:
# Complex types - Complex types help you organize and structure your data in ways that make more sense for the problem that you are hoping
# to solve. They can be structs, arrays or maps.

# Structs 
# We can think of struct as a dataframe within a dataframe
df.selectExpr("struct(Description, InvoiceNo) as complex", "*")
complexDf = df.select(F.struct("Description", "InvoiceNo").alias("complex"))
complexDf.createOrReplaceTempView("complexDf")

# We now have a Dataframe with a column complex which is a struct. We can query it by using a dot
complexDf.select("complex.Description")
complexDf.select(F.col("complex").getField("Description"))
complexDf.select("complex.*")

# Arrays
# Let's first create an array with our data
df.select(F.split(F.col("Description"), " ")).show(2)

(
    df
    .select(
        F.split(F.col("Description"), " ").alias("array_col")
    )
    .selectExpr("array_col[0]").show(2)
)

# Check the array length
df.select(F.size(F.split(F.col("Description"), " "))).show(2)

# Array contains, check if the array contains any value
df.select(F.array_contains(F.split(F.col("Description"), " "), "WHITE")).show(2)

# Explode function allows us to pass an array to a row per item
(
    df
    .withColumn("splitted", F.split(F.col("Description"), " "))
    .withColumn("exploded", F.explode(F.col("splitted")))
    .select("Description", "InvoiceNo", "exploded").show(2)
)

# Maps
# Maps are created using the map functions and key-value pairs of columns
df.select(F.create_map(F.col("Description"), F.col("InvoiceNo")).alias("complex_map")).show(2)

# We can query them by using the key
(
    df
    .select(F.create_map(F.col("Description"), F.col("InvoiceNo")).alias("complex_map"))
    .selectExpr("complex_map['WHITE METAL LANTERN']").show(2)
)

# We can also explode map types, which give us columns
(
    df
    .select(
        F.create_map(F.col("Description"), F.col("InvoiceNo")).alias("complex_map")
    )
    .selectExpr("explode(complex_map)").show(2)
)

+-------------------------+
|split(Description,  , -1)|
+-------------------------+
|     [WHITE, HANGING, ...|
|     [WHITE, METAL, LA...|
+-------------------------+
only showing top 2 rows
+------------+
|array_col[0]|
+------------+
|       WHITE|
|       WHITE|
+------------+
only showing top 2 rows
+-------------------------------+
|size(split(Description,  , -1))|
+-------------------------------+
|                              5|
|                              3|
+-------------------------------+
only showing top 2 rows
+------------------------------------------------+
|array_contains(split(Description,  , -1), WHITE)|
+------------------------------------------------+
|                                            true|
|                                            true|
+------------------------------------------------+
only showing top 2 rows
+--------------------+---------+--------+
|         Description|InvoiceNo|exploded|
+--------------------+---------+--------+
|WHITE HAN

In [35]:
# Working with JSON
jsonDF = (
    spark
    .range(1)
    .selectExpr(
        """
        '{"myJsonKey": {"myJsonValue": [1, 2, 3]}}' as json_string
        """
    )
)

# We can use the get_json_object to inline query a Json object. We can also use json_tuple if this object has only one level of nesting
(
    jsonDF
    .select(
        F.get_json_object(F.col("json_string"), "$.myJsonKey.myJsonValue[1]").alias("column"),
        F.json_tuple(F.col("json_string"), "myJsonKey")
    ).show(2)
)

# We can also turn a struct type into a JSON string by using the to_json method
(
    df
    .selectExpr("(InvoiceNo, Description) as myStruct")
    .select(F.to_json(F.col("myStruct")))
)


# With schema definition
import pyspark.sql.types as T
parseSchema = T.StructType((
    T.StructField("InvoiceNo", F.StringType(), True),
    T.StructField("Description", F.StringType(), True)
))
(
    df
    .selectExpr("(InvoiceNo, Description) as myStruct")
    .select(F.to_json(F.col("myStruct")).alias("newJson"))
    .select(F.from_json(F.col("newJson"), parseSchema), F.col("newJson"))
    .show(2)
)

+------+--------------------+
|column|                  c0|
+------+--------------------+
|     2|{"myJsonValue":[1...|
+------+--------------------+

+--------------------+--------------------+
|  from_json(newJson)|             newJson|
+--------------------+--------------------+
|{536365, WHITE HA...|{"InvoiceNo":"536...|
|{536365, WHITE ME...|{"InvoiceNo":"536...|
+--------------------+--------------------+
only showing top 2 rows


## UDF - User Defined functions
UDF allow you to build your own functions and use them on a dataframe.

After we create the functions, we need to register them with Spark so that we can use them on all our workers machines. Spark will serialize the function on the driver and transfer it over the network to all executor processes.

### Difference between Java/Scala UDFs and Python
Using a UDF behaves differently depending on language:
* Scala/Java UDFs run directly in the JVM, so overhead is relatively low.
* The main downside is they miss Spark's built-in code generation optimizations
* Performance can still degrade if the UDF creates many objects
* Python UDFs incur much higher overhead:
  * Spark launches a Python process on each worker;
  * Data is serialized from JVM format into python-readable format;
  * The function runs row-by-row in Python;
  * Results are serialized back to the JVM for Spark.

Core point: Python UDFs are typically slower because of cross-language process and serialization costs, while Scala/Java UDFs stay in- JVM and are generally cheaper.

![alt text](image.png)

**WARNING**
Python UDFs are expensive mainly because moving data from the JVM to python is costly:
* There is overhead to launching the Python processes;
* The bigger cost is JVM <-> Python serialization;
* Once data is in Python, Spark loses direct memory control for that portion;
* This can cause worker instability or failure when JVM and Python compete for limited memory;

**Recommendation:** prefer Scala/Java UDFs. They usually take only a bit more effort to write but deliver major performance gains, and they can still be called from python


## Usage of UDF's on SQL
After registering UDFs, you can call them from SQL, but type compatibility still matters (the Python example fails because of return-type mismatch).

Key points:

- SQL can invoke registered Scala or Python UDFs.
- If a UDF may return “no value”:
  - Python: return `None`
  - Scala: return `Option`
- You can also define UDF/UDAF-style functions using Hive syntax.
- For Hive-style registration, Spark must be started with Hive support:
  - `SparkSession.builder().enableHiveSupport()`
- Hive SQL function registration supports precompiled Scala/Java classes only, referenced by class name:
  - `CREATE TEMPORARY FUNCTION myFunc AS 'com.organization.hive.udf.FunctionName'`
- Removing `TEMPORARY` makes it a permanent function in the Hive Metastore.

In [41]:
udfExampleDf = spark.range(5).toDF("num")
def power3(double_value):
    return double_value ** 3
power3(2.0)
power3udf = F.udf(power3)

udfExampleDf.select(power3udf(F.col("num"))).show(2)

/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


+-----------+
|power3(num)|
+-----------+
|          0|
|          1|
+-----------+
only showing top 2 rows


In [45]:
# We can register the code in Scala
#spark.udf.register("power3", power3(_:Double):Double)
#udfExampleDF.selectExpr("power3(num)").show(2)
# And then use it on SQL or python, because the UDF was registered on the Spark Session
spark.udf.register("power3py", power3, T.DoubleType())
udfExampleDf.selectExpr("power3py(num)").show(2) # Here we get all nulls because we specify that the return value is a Double


spark.udf.register("power3py_correct", power3, T.IntegerType())
udfExampleDf.selectExpr("power3py_correct(num)").show(2) # now we see values

+-------------+
|power3py(num)|
+-------------+
|         NULL|
|         NULL|
+-------------+
only showing top 2 rows
+---------------------+
|power3py_correct(num)|
+---------------------+
|                    0|
|                    1|
+---------------------+
only showing top 2 rows


26/08/27 08:48:17 WARN SimpleFunctionRegistry: The function system.session.power3py replaced a previously registered function.
